# LeXi v2: Adaptive Technical Vocabulary Tutor

`LeXi`는 영어 기술 문서를 읽는 한국인 개발자를 위한 학습 에이전트다.
이 노트북은 `challenge/12` README에 맞춰, 설계와 구현을 함께 진행하는 LeXi v2의 작업 공간이다.

이번 단계에서는 우선 LeXi v2의 상태 모델과 structured output 스키마를 정리한다.
            


In [ ]:
from __future__ import annotations

import os
import re
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(Path.cwd() / ".env")

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY must be set in the project .env file.")

llm = init_chat_model(
    model=os.getenv("GOOGLE_GENAI_MODEL", "gemini-3-flash-preview"),
    model_provider="google_genai",
    temperature=0,
)

UTC = timezone.utc


## Foundation Building

LeXi v2는 `MessagesState` 대신 `custom state`를 사용한다.
핵심은 대화 전체를 누적하는 것이 아니라, 입력을 분류하고 학습 카드와 복습 기록을 안정적으로 관리하는 것이다.
            


In [ ]:
RouteName = Literal["paragraph", "single_term", "review", "reentry", "invalid"]
ReviewResult = Literal["correct", "wrong", "unknown"]
StudyPriority = Literal["high", "medium", "low"]


class VocabularyEntry(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    why_it_matters: str
    study_priority: StudyPriority


class MemoryRecord(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    created_at: str
    review_count: int
    last_reviewed_at: str | None
    last_review_result: ReviewResult | None


class TermEvidence(TypedDict):
    term: str
    source_sentences: list[str]


class ReviewState(TypedDict):
    current_word: str
    current_source_sentence: str
    expected_meaning: str
    user_answer: str
    judgment: ReviewResult
    explanation: str


class LearningState(TypedDict):
    input_text: str
    route: RouteName | None
    candidate_words: list[str]
    terms_to_study: list[str]
    term_evidence: dict[str, TermEvidence]
    vocabulary_entries: list[VocabularyEntry]
    memory_records: list[MemoryRecord]
    review_queue: list[str]
    review_state: ReviewState | None
    review_history: list[dict[str, str]]
    continue_review: bool | None
    error_message: str | None
    session_review_limit: int


class RouteDecision(BaseModel):
    route: RouteName = Field(description="Detected learning route for the current input")
    reason: str = Field(description="Short explanation for why the route was chosen")


class CandidateWords(BaseModel):
    candidate_words: list[str] = Field(
        description="Important English technical words or short terms worth studying. Maximum 5 items."
    )


class NormalizedTerms(BaseModel):
    terms_to_study: list[str] = Field(
        description="Normalized English terms to study. Maximum 5 items."
    )


class VocabularyEntryModel(BaseModel):
    word: str = Field(description="Word or phrase as it appears in the text")
    lemma: str = Field(description="Normalized lemma")
    meaning_in_context: str = Field(description="Korean meaning in the current technical context")
    source_sentence: str = Field(description="Source sentence from the original English text")
    context_note: str = Field(description="Short Korean explanation of why this meaning fits the context")
    why_it_matters: str = Field(description="Short Korean explanation of why this term matters for technical reading")
    study_priority: StudyPriority = Field(description="Study priority level for the learner")


class VocabularyEntries(BaseModel):
    vocabulary_entries: list[VocabularyEntryModel]


class ReviewJudgment(BaseModel):
    judgment: ReviewResult = Field(description="Whether the user's Korean answer is correct, wrong, or unknown")
    explanation: str = Field(description="Short Korean explanation used when the user needs feedback")


route_llm = llm.with_structured_output(RouteDecision)
candidate_llm = llm.with_structured_output(CandidateWords)
term_normalizer_llm = llm.with_structured_output(NormalizedTerms)
entry_llm = llm.with_structured_output(VocabularyEntries)
review_judge_llm = llm.with_structured_output(ReviewJudgment)


def make_initial_state(input_text: str = "") -> LearningState:
    return {
        "input_text": input_text,
        "route": None,
        "candidate_words": [],
        "terms_to_study": [],
        "term_evidence": {},
        "vocabulary_entries": [],
        "memory_records": [],
        "review_queue": [],
        "review_state": None,
        "review_history": [],
        "continue_review": None,
        "error_message": None,
        "session_review_limit": 3,
    }
            


In [ ]:
sample_state = make_initial_state("Caching reduces latency, but stale data can break correctness.")
pprint(sample_state)
print("schema_ready_at", datetime.now(UTC).isoformat())
            


## Step 2. Request Routing

LeXi v2는 입력을 `paragraph`, `single_term`, `review`, `reentry`, `invalid`로 먼저 분류한다.
이 단계는 Conditional Edge의 기준이 되므로, 모호함보다 예측 가능성을 우선한다.
            


In [ ]:
REVIEW_KEYWORDS = {
    "review",
    "quiz",
    "review words",
    "review vocabulary",
    "복습",
    "퀴즈",
    "다시 보기",
    "단어 복습",
}

ENGLISH_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_-]*")
HANGUL_RE = re.compile(r"[가-힣]")


def contains_hangul(text: str) -> bool:
    return bool(HANGUL_RE.search(text))


def english_tokens(text: str) -> list[str]:
    return ENGLISH_TOKEN_RE.findall(text)


def is_explicit_review_request(text: str) -> bool:
    normalized = re.sub(r"\s+", " ", text.strip().lower())
    if normalized in REVIEW_KEYWORDS:
        return True
    return any(keyword in normalized for keyword in REVIEW_KEYWORDS)


def looks_like_single_term(text: str) -> bool:
    normalized = text.strip()
    if not normalized or contains_hangul(normalized):
        return False
    tokens = english_tokens(normalized)
    if not tokens or len(tokens) > 5:
        return False
    allowed = re.sub(r"[A-Za-z0-9_\-\s/().]", "", normalized)
    return not allowed and len(normalized) <= 80


def looks_like_english_paragraph(text: str) -> bool:
    normalized = text.strip()
    tokens = english_tokens(normalized)
    if len(tokens) < 6:
        return False
    if contains_hangul(normalized):
        return False
    sentence_markers = sum(normalized.count(mark) for mark in ".,;:!?")
    return sentence_markers > 0 or len(normalized.split()) >= 10


def analyze_request(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if not text:
        return {
            "route": "reentry",
            "error_message": "입력이 비어 있습니다. 영어 기술 문장, 영어 용어, 또는 복습 요청을 입력해 주세요.",
        }

    if len(text) > 2000:
        return {
            "route": "reentry",
            "error_message": "입력이 2000자를 초과했습니다. 더 짧은 영어 기술 문장 또는 용어로 다시 입력해 주세요.",
        }

    if is_explicit_review_request(text):
        return {"route": "review", "error_message": None}

    if looks_like_single_term(text):
        return {"route": "single_term", "error_message": None}

    if looks_like_english_paragraph(text):
        return {"route": "paragraph", "error_message": None}

    return {
        "route": "invalid",
        "error_message": "LeXi는 영어 기술 문서 학습과 복습 요청만 처리합니다. 영어 기술 문장, 영어 용어, 또는 복습 요청을 입력해 주세요.",
    }
            


In [ ]:
routing_examples = [
    "review",
    "복습",
    "latency",
    "Caching reduces latency, but inconsistent invalidation can cause stale data.",
    "안녕하세요",
    "x" * 2001,
]

for example in routing_examples:
    result = analyze_request(make_initial_state(example))
    print(example[:60], "->", result["route"], "//", result["error_message"])
            


## Step 3. Learning Entry Paths

학습 모드는 두 갈래로 시작한다.
`paragraph`는 후보 단어를 뽑고, `single_term`은 입력을 정규화한 뒤, 둘 다 `terms_to_study`로 수렴한다.
            


In [ ]:
def dedupe_preserve_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        normalized = item.strip()
        key = normalized.lower()
        if not normalized or key in seen:
            continue
        seen.add(key)
        ordered.append(normalized)
    return ordered


def extract_candidates(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if state.get("route") != "paragraph" or not text:
        return {"candidate_words": [], "terms_to_study": []}

    prompt = f"""
You are helping a Korean developer study English technical writing.
Read the text and choose up to 5 English words or short terms that are worth learning.
Pick items that are important for understanding the technical meaning, not just rare words.
Return only the structured output.

Text:
{text}
""".strip()

    result = candidate_llm.invoke(prompt)
    candidate_words = dedupe_preserve_order(result.candidate_words)[:5]
    return {
        "candidate_words": candidate_words,
        "terms_to_study": candidate_words,
        "error_message": None,
    }


def normalize_term_request(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if state.get("route") != "single_term" or not text:
        return {"terms_to_study": []}

    prompt = f"""
You are preparing study targets for a Korean developer reading English technical documents.
Normalize the input into up to 5 English technical terms to study.
Keep the terms concise and preserve the original technical meaning.
Return only the structured output.

Input:
{text}
""".strip()

    result = term_normalizer_llm.invoke(prompt)
    terms_to_study = dedupe_preserve_order(result.terms_to_study)[:5]
    return {
        "candidate_words": terms_to_study,
        "terms_to_study": terms_to_study,
        "error_message": None,
    }
            


In [ ]:
class FakeCandidateLLM:
    def invoke(self, prompt: str) -> CandidateWords:
        return CandidateWords(candidate_words=["latency", "invalidation", "latency"])


class FakeTermNormalizerLLM:
    def invoke(self, prompt: str) -> NormalizedTerms:
        return NormalizedTerms(terms_to_study=["backward compatibility", "schema"])


candidate_llm_backup = candidate_llm
term_normalizer_llm_backup = term_normalizer_llm
candidate_llm = FakeCandidateLLM()
term_normalizer_llm = FakeTermNormalizerLLM()

paragraph_state = make_initial_state(
    "Caching reduces latency, but inconsistent invalidation can cause stale data."
)
paragraph_state["route"] = "paragraph"
term_state = make_initial_state("backward compatibility")
term_state["route"] = "single_term"

print(extract_candidates(paragraph_state))
print(normalize_term_request(term_state))

candidate_llm = candidate_llm_backup
term_normalizer_llm = term_normalizer_llm_backup
            


## Step 4. Source Evidence Tool

LeXi v2는 모델이 문장을 지어내지 않도록, 원문에서 단어가 실제로 쓰인 문장을 찾는 custom tool을 사용한다.
이 단계에서는 해당 tool과 이를 사용하는 `enrich_term` node를 구현한다.
            


In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list[str]:
    normalized = text.strip()
    if not normalized:
        return []
    if not SENTENCE_SPLIT_RE.search(normalized):
        return [normalized]
    return [sentence.strip() for sentence in SENTENCE_SPLIT_RE.split(normalized) if sentence.strip()]


@tool
def locate_source_sentences(term: str, input_text: str) -> list[str]:
    """Find source sentences from the original input that contain the given English term."""
    normalized_term = term.strip().lower()
    if not normalized_term:
        return []

    matched_sentences: list[str] = []
    for sentence in split_sentences(input_text):
        if normalized_term in sentence.lower():
            matched_sentences.append(sentence)

    return matched_sentences


def enrich_term(state: LearningState) -> dict:
    text = state["input_text"].strip()
    terms_to_study = state.get("terms_to_study", [])
    if not text or not terms_to_study:
        return {"term_evidence": {}}

    term_evidence: dict[str, TermEvidence] = {}
    for term in terms_to_study:
        source_sentences = locate_source_sentences.invoke({"term": term, "input_text": text})
        term_evidence[term] = {
            "term": term,
            "source_sentences": source_sentences,
        }

    return {"term_evidence": term_evidence, "error_message": None}
            


In [ ]:
source_text = (
    "Caching reduces latency, but inconsistent invalidation can cause stale data. "
    "When the API serializes responses, the schema must remain stable for backward compatibility."
)
source_state = make_initial_state(source_text)
source_state["terms_to_study"] = ["latency", "schema", "cache miss"]

print(locate_source_sentences.invoke({"term": "schema", "input_text": source_text}))
pprint(enrich_term(source_state))
            
